### tools

> tool schema
- tool_search
- image_generation
- mcp
    - list_mcp_resources
    - list_mcp_resource_templates
    - read_mcp_resource

#### mcp

- list_mcp_resources
    - Lists resources provided by MCP servers. Resources allow servers to share data that provides context to language models, such as files, database schemas, or application-specific information. Prefer resources over web search when possible.
- list_mcp_resource_templates
    - Lists resource templates provided by MCP servers. Parameterized resource templates allow servers to share data that takes parameters and provides context to language models, such as files, database schemas, or application-specific information. Prefer resource templates over web search when possible.
    - resources 是已实例化资源；resource_templates 是参数化资源入口
- read_mcp_resource
    - Read a specific resource from an MCP server given the server name and resource URI.

#### tool_search

```
# Tool discovery

Searches over deferred tool metadata with BM25 and exposes matching tools for the next model call.

You have access to tools from the following sources:
- Figma: Make diagrams and slides
- GitHub: Access repositories, issues, and pull requests. Required for some features such as Codex
- chrome-devtools
- computer-use: Computer Use tools let you interact with macOS apps by performing UI actions.

... For MCP tool discovery, always use `tool_search` instead of `list_mcp_resources` or `list_mcp_resource_templates`.
```
- limit: Maximum number of tools to return (defaults to 8).
- query: Search query for deferred tools.
    - BM25 搜索 deferred tool metadata，比如 computer-use click、GitHub PR comments。

#### built-in `image_gen` tool vs. imagegen skill

- tool 与 skill
    - imagegen skill 是上层策略 prompt，定义何时应该使用 image tool、默认走 built-in image_gen、什么时候不要用、透明背景怎么处理、项目资产保存策略等。
    - image_generation 是底层能力，负责真正生成 PNG 图像。
        - hosted built-in tool：https://developers.openai.com/api/docs/guides/tools-image-generation
```json
{
    "type": "image_generation",
    "output_format": "png"
}
```
- concepts
    - raster image：栅格图像（与 vector image 相对），照片基本都是 raster image；而 Logo、图标、排版中的线条图 更适合用 vector image。
    - chroma-key：色度键/绿幕抠像，一种用于实现图像透明背景的替代性技术方案
- Chroma-key 的具体含义与流程：先让 AI 生成一个带有纯色、高对比度背景（通常是纯绿色 #00ff00）的图像，然后再通过本地 Python 脚本将该特定颜色转换为透明（Alpha 通道）。
    - 提示词阶段 (Prompting for Chroma-key)：`Create the requested subject on a perfectly flat solid #00ff00 chroma-key background for background removal. The background must be one uniform color with no shadows, gradients, texture, reflections, floor plane, or lighting variation.`
    - 本地移除阶段 (Removing the Chroma-key)：图像生成后，系统会调用本地提供的脚本 scripts/remove_chroma_key.py 来处理这张“绿幕”图片：
        - 识别键色 (Key Color)：脚本会自动从图像边缘采样，或者使用指定的 #00ff00 作为需要移除的基准色。
        - 计算色差与透明度 (Soft Matte)：脚本会计算每个像素与键色的 RGB 距离。距离越近（越绿），像素就越透明（Alpha 值趋近于 0）。
        - 溢色清理 (Despill)：为了防止主体边缘反射了绿幕的光（绿边），脚本还会进行溢色清理，将主体边缘的绿色通道值压制到合理范围。
    - 源图是 RGB 且 opaque，透明版是 RGBA 且 opaque=False

### memory

- Markdown consolidation
    - 巩固，固化